# 98. The Green Vehicle Routing Problem

## Tier 9: Quantum Leap (QUBO Formulation)

### Key assumptions
- Formulate G-VRP as a Quadratic Unconstrained Binary Optimization (QUBO) problem
- Use quantum annealing or quantum-inspired classical simulation
- Map routing decisions to binary variables with penalty terms for constraints
- Leverage quantum computing for exponential speedup potential
- Hybrid classical-quantum approach for practical implementation

### Approach (step-by-step)
1. **QUBO Formulation**: Convert routing problem to QUBO format
2. **Binary Encoding**: Map routing decisions to binary variables
3. **Hamiltonian Construction**: Create objective and constraint Hamiltonian components
4. **Quantum Optimization**: Use quantum annealing to find ground state
5. **Classical Simulation**: Use classical solvers for comparison
6. **Performance Analysis**: Compare quantum vs classical approaches

### What to look for in the results
- QUBO matrix structure and sparsity patterns
- Quantum annealing convergence behavior and ground state quality
- Classical-quantum hybrid performance comparison
- Computational complexity analysis and speedup potential
- Constraint satisfaction and penalty effectiveness

### Concrete example (from the source)
We'll implement a QUBO formulation for a small G-VRP:
- 2 customers with depot (0, C1, C2)
- Binary variables: x_{i,p,k} = 1 if vehicle k visits customer i at position p
- Distance matrix and emissions matrix with realistic values
- Constraint Hamiltonians with penalty coefficients
- Quantum annealing with temperature schedule
- Classical simulation for validation

### Visualization(s)
- QUBO matrix heat map visualization
- Quantum annealing convergence curves
- Route comparison: quantum vs classical solutions
- Constraint violation analysis
- Performance metrics and speedup analysis

### Why this Tier exists vs earlier Tiers
Tier 9 provides quantum computing capabilities that address computational limitations of classical methods:
- **Exponential Speedup**: Potential for exponential computational advantage
- **Quantum Parallelism**: Simultaneous evaluation of multiple solutions
- **Hardware Acceleration**: Leverages quantum hardware for optimization
- **Future-Proofing**: Prepares for quantum computing era
- **Hybrid Approach**: Combines classical and quantum strengths

### Pros / Cons vs Earlier Tiers
**Advantages over Tiers 1-8:**
- Potential exponential speedup for large problems
- Natural handling of combinatorial optimization
- Quantum parallelism for simultaneous solution evaluation
- Hardware acceleration through quantum processors
- Novel computational paradigm beyond classical limits
- Future-proof technology for next-generation computing

**Disadvantages:**
- Limited quantum hardware availability
- Noise and error correction challenges
- Complex QUBO formulation requirements
- Hybrid approaches needed for practical problems
- Limited problem size due to hardware constraints
- Requires specialized quantum expertise

### When to use this Tier
- **Large-Scale Problems**: When classical methods become computationally intractable
- **Combinatorial Optimization**: Problems with exponential solution spaces
- **Research Applications**: Exploring quantum computing capabilities
- **Future-Proofing**: Preparing for quantum computing adoption
- **Hybrid Solutions**: Combining classical and quantum approaches
- **High-Value Optimization**: When computational speedup provides significant value

In [1]:
# Import required libraries for QUBO Formulation
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import minimize
import random
from dataclasses import dataclass
from typing import List, Dict, Tuple, Optional
import warnings
warnings.filterwarnings('ignore')

# Set plotting style for professional visualizations
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Set random seeds for reproducibility
np.random.seed(42)
random.seed(42)

In [ ]:
# Define QUBO components for Green VRP

@dataclass
class QUBOInstance:
    """QUBO instance for Green VRP"""
    n_customers: int
    n_vehicles: int
    n_positions: int
    distances: np.ndarray
    emissions: np.ndarray
    vehicle_capacities: List[int]
    customer_demands: List[int]
    
    def __post_init__(self):
        # Calculate total number of binary variables
        self.n_variables = self.n_customers * self.n_positions * self.n_vehicles
        
    def get_variable_index(self, customer: int, position: int, vehicle: int) -> int:
        """Get linear index for 3D binary variable"""
        return customer * self.n_positions * self.n_vehicles + position * self.n_vehicles + vehicle


@dataclass
class QUBOSolution:
    """Solution to QUBO problem"""
    binary_vector: np.ndarray
    routes: List[List[int]]
    total_cost: float
    total_emissions: float
    total_distance: float
    energy: float
    is_feasible: bool
    constraint_violations: Dict[str, float]


class QUBO_GVRP:
    """QUBO formulation for Green Vehicle Routing Problem"""
    
    def __init__(self, instance: QUBOInstance):
        self.instance = instance
        
        # QUBO parameters
        self.Q_matrix = None
        self.penalty_weights = {
            'assignment': 1000.0,    # Each customer assigned exactly once
            'position': 1000.0,      # Each position occupied by at most one customer
            'vehicle': 1000.0,       # Vehicle capacity constraints
            'continuity': 1000.0     # Route continuity constraints
        }
        
        # Multi-objective weights
        self.cost_weight = 0.5
        self.emissions_weight = 0.5
        
    def build_qubo_matrix(self) -> np.ndarray:
        """Build QUBO matrix for Green VRP"""
        n = self.instance.n_variables
        Q = np.zeros((n, n))
        
        print("Building QUBO matrix...")
        print(f"  Variables: {n}")
        print(f"  Customers: {self.instance.n_customers}")
        print(f"  Vehicles: {self.instance.n_vehicles}")
        print(f"  Positions: {self.instance.n_positions}")
        
        # 1. Objective function: minimize cost + emissions
        self._add_objective_terms(Q)
        
        # 2. Constraint penalties
        self._add_assignment_constraints(Q)
        self._add_position_constraints(Q)
        self._add_capacity_constraints(Q)
        self._add_continuity_constraints(Q)
        
        self.Q_matrix = Q
        print(f"QUBO matrix built: {n}x{n}")
        print(f"  Non-zero elements: {np.count_nonzero(Q)}")
        print(f"  Sparsity: {(1 - np.count_nonzero(Q) / (n * n)) * 100:.1f}%")
        
        return Q
    
    def _add_objective_terms(self, Q: np.ndarray):
        """Add cost and emissions objective terms to QUBO matrix"""
        print("  Adding objective terms...")
        
        for customer in range(self.instance.n_customers):
            for pos1 in range(self.instance.n_positions):
                for vehicle1 in range(self.instance.n_vehicles):
                    idx1 = self.instance.get_variable_index(customer, pos1, vehicle1)
                    
                    # Linear terms (cost for visiting customer)
                    # Cost from depot to customer (if first position)
                    if pos1 == 0:
                        distance = self.instance.distances[0, customer + 1]
                        emissions = self.instance.emissions[0, customer + 1]
                        cost = self.cost_weight * distance + self.emissions_weight * emissions
                        Q[idx1, idx1] += cost
                    
                    # Quadratic terms (cost between consecutive customers)
                    for customer2 in range(self.instance.n_customers):
                        for pos2 in range(self.instance.n_positions):
                            for vehicle2 in range(self.instance.n_vehicles):
                                if pos1 + 1 == pos2 and vehicle1 == vehicle2:
                                    idx2 = self.instance.get_variable_index(customer2, pos2, vehicle2)
                                    
                                    # Cost from customer1 to customer2
                                    distance = self.instance.distances[customer + 1, customer2 + 1]
                                    emissions = self.instance.emissions[customer + 1, customer2 + 1]
                                    cost = self.cost_weight * distance + self.emissions_weight * emissions
                                    
                                    Q[idx1, idx2] += cost
                                    Q[idx2, idx1] += cost  # Symmetric
        
        print("    Objective terms added")
    
    def _add_assignment_constraints(self, Q: np.ndarray):
        """Add assignment constraints: each customer assigned exactly once"""
        print("  Adding assignment constraints...")
        
        penalty = self.penalty_weights['assignment']
        
        for customer in range(self.instance.n_customers):
            # Each customer must be assigned to exactly one position and vehicle
            indices = []
            for pos in range(self.instance.n_positions):
                for vehicle in range(self.instance.n_vehicles):
                    idx = self.instance.get_variable_index(customer, pos, vehicle)
                    indices.append(idx)
            
            # Add penalty: (sum(x_i) - 1)^2 = sum(x_i^2) + 2*sum(x_i*x_j) - 2*sum(x_i) + 1
            for idx in indices:
                Q[idx, idx] += penalty  # x_i^2 term
                Q[idx, idx] -= 2 * penalty  # -2*x_i term
            
            for i in range(len(indices)):
                for j in range(i + 1, len(indices)):
                    idx_i, idx_j = indices[i], indices[j]
                    Q[idx_i, idx_j] += 2 * penalty  # 2*x_i*x_j term
                    Q[idx_j, idx_i] += 2 * penalty  # Symmetric
        
        print("    Assignment constraints added")
    
    def _add_position_constraints(self, Q: np.ndarray):
        """Add position constraints: each position occupied by at most one customer"""
        print("  Adding position constraints...")
        
        penalty = self.penalty_weights['position']
        
        for vehicle in range(self.instance.n_vehicles):
            for pos in range(self.instance.n_positions):
                # Each position can be occupied by at most one customer
                indices = []
                for customer in range(self.instance.n_customers):
                    idx = self.instance.get_variable_index(customer, pos, vehicle)
                    indices.append(idx)
                
                # Add penalty: sum(x_i*x_j) for all i != j
                for i in range(len(indices)):
                    for j in range(i + 1, len(indices)):
                        idx_i, idx_j = indices[i], indices[j]
                        Q[idx_i, idx_j] += penalty  # x_i*x_j term
                        Q[idx_j, idx_i] += penalty  # Symmetric
        
        print("    Position constraints added")
    
    def _add_capacity_constraints(self, Q: np.ndarray):
        """Add vehicle capacity constraints"""
        print("  Adding capacity constraints...")
        
        penalty = self.penalty_weights['vehicle']
        
        for vehicle in range(self.instance.n_vehicles):
            capacity = self.instance.vehicle_capacities[vehicle]
            
            # Calculate total demand assigned to vehicle
            demand_terms = []
            for customer in range(self.instance.n_customers):
                demand = self.instance.customer_demands[customer]
                for pos in range(self.instance.n_positions):
                    idx = self.instance.get_variable_index(customer, pos, vehicle)
                    demand_terms.append((idx, demand))
            
            # Add penalty for exceeding capacity
            for idx, demand in demand_terms:
                Q[idx, idx] += penalty * demand * demand  # demand^2 * x_i^2
            
            for i in range(len(demand_terms)):
                for j in range(i + 1, len(demand_terms)):
                    idx_i, demand_i = demand_terms[i]
                    idx_j, demand_j = demand_terms[j]
                    Q[idx_i, idx_j] += 2 * penalty * demand_i * demand_j  # 2*demand_i*demand_j*x_i*x_j
                    Q[idx_j, idx_i] += 2 * penalty * demand_i * demand_j  # Symmetric
        
        print("    Capacity constraints added")
    
    def _add_continuity_constraints(self, Q: np.ndarray):
        """Add route continuity constraints"""
        print("  Adding continuity constraints...")
        
        penalty = self.penalty_weights['continuity']
        
        # Simplified continuity: if position p is occupied, position p+1 should be occupied by the same vehicle
        for vehicle in range(self.instance.n_vehicles):
            for pos in range(self.instance.n_positions - 1):
                # Get all customers that can occupy position pos and pos+1
                pos_indices = []
                next_pos_indices = []
                
                for customer in range(self.instance.n_customers):
                    pos_idx = self.instance.get_variable_index(customer, pos, vehicle)
                    next_pos_idx = self.instance.get_variable_index(customer, pos + 1, vehicle)
                    pos_indices.append(pos_idx)
                    next_pos_indices.append(next_pos_idx)
                
                # Add penalty for discontinuity
                for i in range(len(pos_indices)):
                    for j in range(len(next_pos_indices)):
                        if i != j:  # Different customers in consecutive positions
                            idx_i, idx_j = pos_indices[i], next_pos_indices[j]
                            Q[idx_i, idx_j] += penalty  # x_i * x_j penalty
                            Q[idx_j, idx_i] += penalty  # Symmetric
        
        print("    Continuity constraints added")
    
    def calculate_energy(self, binary_vector: np.ndarray) -> float:
        """Calculate energy of binary vector"""
        if self.Q_matrix is None:
            raise ValueError("QUBO matrix not built")
        
        return binary_vector @ self.Q_matrix @ binary_vector
    
    def decode_solution(self, binary_vector: np.ndarray) -> QUBOSolution:
        """Decode binary vector to routing solution"""
        routes = []
        total_cost = 0.0
        total_emissions = 0.0
        total_distance = 0.0
        
        # Extract routes from binary vector
        for vehicle in range(self.instance.n_vehicles):
            route = []
            for pos in range(self.instance.n_positions):
                for customer in range(self.instance.n_customers):
                    idx = self.instance.get_variable_index(customer, pos, vehicle)
                    if binary_vector[idx] == 1:
                        route.append(customer + 1)  # Convert to 1-based indexing
            
            if route:  # Only add non-empty routes
                routes.append(route)
                
                # Calculate route metrics
                # Add depot at start and end
                full_route = [0] + route + [0]
                
                for i in range(len(full_route) - 1):
                    from_node = full_route[i]
                    to_node = full_route[i + 1]
                    
                    distance = self.instance.distances[from_node][to_node]
                    emissions = self.instance.emissions[from_node][to_node]
                    
                    total_distance += distance
                    total_emissions += emissions
                    total_cost += distance * 2.0  # $2 per km
        
        # Calculate energy
        energy = self.calculate_energy(binary_vector)
        
        # Check feasibility
        constraint_violations = self._check_constraints(binary_vector)
        is_feasible = len(constraint_violations) == 0
        
        return QUBOSolution(
            binary_vector=binary_vector,
            routes=routes,
            total_cost=total_cost,
            total_emissions=total_emissions,
            total_distance=total_distance,
            energy=energy,
            is_feasible=is_feasible,
            constraint_violations=constraint_violations
        )
    
    def _check_constraints(self, binary_vector: np.ndarray) -> Dict[str, float]:
        """Check constraint violations"""
        violations = {}
        
        # Check assignment constraints
        for customer in range(self.instance.n_customers):
            assignments = 0
            for pos in range(self.instance.n_positions):
                for vehicle in range(self.instance.n_vehicles):
                    idx = self.instance.get_variable_index(customer, pos, vehicle)
                    assignments += binary_vector[idx]
            
            if assignments != 1:
                violations[f'customer_{customer}_assignment'] = abs(assignments - 1)
        
        # Check capacity constraints
        for vehicle in range(self.instance.n_vehicles):
            total_demand = 0
            for customer in range(self.instance.n_customers):
                demand = self.instance.customer_demands[customer]
                for pos in range(self.instance.n_positions):
                    idx = self.instance.get_variable_index(customer, pos, vehicle)
                    total_demand += binary_vector[idx] * demand
            
            if total_demand > self.instance.vehicle_capacities[vehicle]:
                violations[f'vehicle_{vehicle}_capacity'] = total_demand - self.instance.vehicle_capacities[vehicle]
        
        return violations

In [ ]:
# Setup the QUBO problem for Green VRP
print("=" * 60)
print("GREEN VRP - QUBO FORMULATION SETUP")
print("=" * 60)

# Define problem parameters (small instance for demo)
# Nodes: 0=Depot, 1-2=Customers
distances = np.array([
    [0, 10, 12],    # From Depot
    [10, 0, 8],     # Customer 1
    [12, 8, 0]      # Customer 2
])

emissions = np.array([
    [0, 6, 10],     # From Depot
    [6, 0, 5],      # Customer 1
    [10, 5, 0]     # Customer 2
])

# Create QUBO instance
instance = QUBOInstance(
    n_customers=2,
    n_vehicles=2,
    n_positions=2,
    distances=distances,
    emissions=emissions,
    vehicle_capacities=[50, 50],
    customer_demands=[20, 30]
)

print(f"QUBO instance created:")
print(f"  Customers: {instance.n_customers}")
print(f"  Vehicles: {instance.n_vehicles}")
print(f"  Positions: {instance.n_positions}")
print(f"  Binary variables: {instance.n_variables}")
print(f"  Vehicle capacities: {instance.vehicle_capacities}")
print(f"  Customer demands: {instance.customer_demands}")

print(f"\nDistance matrix:")
print(distances)

print(f"\nEmissions matrix:")
print(emissions)

# Create QUBO solver
qubo_solver = QUBO_GVRP(instance)

print(f"\nQUBO solver created:")
print(f"  Penalty weights: {qubo_solver.penalty_weights}")
print(f"  Cost weight: {qubo_solver.cost_weight}")
print(f"  Emissions weight: {qubo_solver.emissions_weight}")

print(f"\nReady for QUBO formulation...")

In [ ]:
# Build QUBO matrix
Q_matrix = qubo_solver.build_qubo_matrix()

print(f"\n" + "=" * 60)
print("QUBO MATRIX ANALYSIS")
print("=" * 60)

print(f"\nQUBO matrix properties:")
print(f"  Shape: {Q_matrix.shape}")
print(f"  Non-zero elements: {np.count_nonzero(Q_matrix)}")
print(f"  Sparsity: {(1 - np.count_nonzero(Q_matrix) / (Q_matrix.shape[0] * Q_matrix.shape[1])) * 100:.1f}%")
print(f"  Symmetric: {np.allclose(Q_matrix, Q_matrix.T)}")
print(f"  Min value: {Q_matrix.min():.2f}")
print(f"  Max value: {Q_matrix.max():.2f}")
print(f"  Mean value: {Q_matrix.mean():.2f}")

# Display a portion of the QUBO matrix
print(f"\nQUBO matrix (first 8x8 elements):")
print(Q_matrix[:8, :8])

# Calculate theoretical minimum energy (if all variables = 0)
zero_vector = np.zeros(instance.n_variables)
zero_energy = qubo_solver.calculate_energy(zero_vector)
print(f"\nZero vector energy: {zero_energy:.2f}")

# Test a simple valid solution
test_vector = np.zeros(instance.n_variables)
# Customer 0 at position 0, vehicle 0
test_vector[instance.get_variable_index(0, 0, 0)] = 1
# Customer 1 at position 0, vehicle 1
test_vector[instance.get_variable_index(1, 0, 1)] = 1

test_energy = qubo_solver.calculate_energy(test_vector)
test_solution = qubo_solver.decode_solution(test_vector)

print(f"\nTest solution:")
print(f"  Energy: {test_energy:.2f}")
print(f"  Routes: {test_solution.routes}")
print(f"  Cost: ${test_solution.total_cost:.2f}")
print(f"  Emissions: {test_solution.total_emissions:.2f} kg CO₂")
print(f"  Feasible: {test_solution.is_feasible}")
print(f"  Constraint violations: {test_solution.constraint_violations}")

In [ ]:
# Implement quantum-inspired classical solver
class QuantumInspiredSolver:
    """Quantum-inspired classical solver for QUBO"""
    
    def __init__(self, qubo_solver: QUBO_GVRP):
        self.qubo_solver = qubo_solver
        
    def simulated_annealing(self, max_iterations: int = 1000, 
                           initial_temp: float = 10.0, 
                           final_temp: float = 0.01) -> QUBOSolution:
        """Simulated annealing solver"""
        print("\n=== Quantum-Inspired Simulated Annealing ===")
        print(f"Max iterations: {max_iterations}")
        print(f"Temperature range: {initial_temp} -> {final_temp}")
        
        n = self.qubo_solver.instance.n_variables
        
        # Initialize random solution
        current_vector = np.random.randint(0, 2, n)
        current_energy = self.qubo_solver.calculate_energy(current_vector)
        
        best_vector = current_vector.copy()
        best_energy = current_energy
        
        # Temperature schedule
        temp_schedule = np.linspace(initial_temp, final_temp, max_iterations)
        
        energy_history = [current_energy]
        
        for iteration in range(max_iterations):
            temp = temp_schedule[iteration]
            
            # Propose new solution (flip one random bit)
            new_vector = current_vector.copy()
            flip_idx = np.random.randint(0, n)
            new_vector[flip_idx] = 1 - new_vector[flip_idx]
            
            new_energy = self.qubo_solver.calculate_energy(new_vector)
            
            # Accept or reject
            delta_energy = new_energy - current_energy
            
            if delta_energy < 0 or np.random.random() < np.exp(-delta_energy / temp):
                current_vector = new_vector
                current_energy = new_energy
                
                if current_energy < best_energy:
                    best_vector = current_vector.copy()
                    best_energy = current_energy
            
            energy_history.append(current_energy)
            
            # Progress reporting
            if (iteration + 1) % 100 == 0:
                print(f"Iteration {iteration + 1}/{max_iterations}")
                print(f"  Current energy: {current_energy:.2f}")
                print(f"  Best energy: {best_energy:.2f}")
                print(f"  Temperature: {temp:.3f}")
        
        best_solution = self.qubo_solver.decode_solution(best_vector)
        
        print(f"\nSimulated annealing complete:")
        print(f"  Best energy: {best_energy:.2f}")
        print(f"  Best solution cost: ${best_solution.total_cost:.2f}")
        print(f"  Best solution emissions: {best_solution.total_emissions:.2f} kg CO₂")
        print(f"  Feasible: {best_solution.is_feasible}")
        
        return best_solution, energy_history
    
    def quantum_approximate_optimization(self, layers: int = 3, 
                                         max_iterations: int = 100) -> QUBOSolution:
        """Quantum Approximate Optimization Algorithm (QAOA) simulation"""
        print("\n=== Quantum Approximate Optimization (QAOA) ===")
        print(f"Layers: {layers}")
        print(f"Max iterations: {max_iterations}")
        
        n = self.qubo_solver.instance.n_variables
        
        # Initialize parameters
        gamma = np.random.uniform(0, 2*np.pi, layers)
        beta = np.random.uniform(0, np.pi, layers)
        
        best_vector = None
        best_energy = float('inf')
        
        for iteration in range(max_iterations):
            # Initialize quantum state (simplified as probability distribution)
            probabilities = np.ones(n) / n  # Equal superposition
            
            # Apply QAOA layers
            for layer in range(layers):
                # Problem unitary (exp(-i*gamma*H))
                # Mixing unitary (exp(-i*beta*X))
                # Simplified: just modify probabilities
                for i in range(n):
                    # Problem Hamiltonian effect
                    for j in range(n):
                        if i != j:
                            Q_ij = self.qubo_solver.Q_matrix[i, j]
                            probabilities[i] *= np.exp(-gamma[layer] * Q_ij * 0.01)
                    
                    # Mixing Hamiltonian effect
                    probabilities[i] = 0.5 * (1 + np.sin(2 * beta[layer]))
            
            # Normalize probabilities
            probabilities /= np.sum(probabilities)
            
            # Sample from distribution
            sampled_vector = np.random.random(n) < probabilities
            sampled_energy = self.qubo_solver.calculate_energy(sampled_vector)
            
            if sampled_energy < best_energy:
                best_vector = sampled_vector.copy()
                best_energy = sampled_energy
            
            # Update parameters (gradient descent - simplified)
            gamma += np.random.uniform(-0.1, 0.1, layers)
            beta += np.random.uniform(-0.1, 0.1, layers)
            
            # Progress reporting
            if (iteration + 1) % 25 == 0:
                print(f"Iteration {iteration + 1}/{max_iterations}")
                print(f"  Best energy: {best_energy:.2f}")
        
        best_solution = self.qubo_solver.decode_solution(best_vector)
        
        print(f"\nQAOA complete:")
        print(f"  Best energy: {best_energy:.2f}")
        print(f"  Best solution cost: ${best_solution.total_cost:.2f}")
        print(f"  Best solution emissions: {best_solution.total_emissions:.2f} kg CO₂")
        print(f"  Feasible: {best_solution.is_feasible}")
        
        return best_solution

# Create solver and run optimization
solver = QuantumInspiredSolver(qubo_solver)

# Run simulated annealing
sa_solution, sa_energy_history = solver.simulated_annealing(
    max_iterations=1000,
    initial_temp=10.0,
    final_temp=0.01
)

# Run QAOA
qaoa_solution = solver.quantum_approximate_optimization(
    layers=3,
    max_iterations=100
)

In [ ]:
# Generate comprehensive visualizations for QUBO analysis
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('QUBO Formulation - Quantum-Inspired Green VRP', fontsize=16, fontweight='bold')

# 1. QUBO Matrix Heat Map
ax1 = axes[0, 0]
im1 = ax1.imshow(Q_matrix, cmap='viridis', aspect='auto')
ax1.set_title('QUBO Matrix Heat Map')
ax1.set_xlabel('Variable Index')
ax1.set_ylabel('Variable Index')
plt.colorbar(im1, ax=ax1, label='Q Value')

# 2. Simulated Annealing Convergence
ax2 = axes[0, 1]
iterations = range(len(sa_energy_history))
ax2.plot(iterations, sa_energy_history, 'b-', linewidth=2, alpha=0.7)
ax2.set_xlabel('Iteration')
ax2.set_ylabel('Energy')
ax2.set_title('Simulated Annealing Convergence')
ax2.grid(True, alpha=0.3)

# Add convergence annotation
min_energy_idx = np.argmin(sa_energy_history)
ax2.annotate(f'Min: {sa_energy_history[min_energy_idx]:.2f}', 
            xy=(min_energy_idx, sa_energy_history[min_energy_idx]),
            xytext=(min_energy_idx + 50, sa_energy_history[min_energy_idx] + 10),
            arrowprops=dict(arrowstyle='->', color='red'),
            fontsize=10, color='red')

# 3. Solution Comparison
ax3 = axes[0, 2]
methods = ['Simulated Annealing', 'QAOA']
costs = [sa_solution.total_cost, qaoa_solution.total_cost]
emissions = [sa_solution.total_emissions, qaoa_solution.total_emissions]
energies = [sa_solution.energy, qaoa_solution.energy]

x = np.arange(len(methods))
width = 0.25

ax3.bar(x - width, costs, width, label='Cost ($)', alpha=0.7)
ax3.bar(x, emissions, width, label='Emissions (kg CO₂)', alpha=0.7)
ax3.bar(x + width, energies, width, label='Energy', alpha=0.7)

ax3.set_xlabel('Method')
ax3.set_ylabel('Value')
ax3.set_title('Solution Comparison')
ax3.set_xticks(x)
ax3.set_xticklabels(methods)
ax3.legend()
ax3.grid(True, alpha=0.3)

# 4. Route Visualization
ax4 = axes[1, 0]
# Plot depot and customers
ax4.scatter(0, 0, s=200, c='red', marker='s', label='Depot', zorder=5)
ax4.scatter(10, 0, s=100, c='blue', marker='o', label='Customer 1', zorder=4)
ax4.scatter(12, 8, s=100, c='green', marker='o', label='Customer 2', zorder=4)

# Plot SA solution routes
colors = ['blue', 'green']
for i, route in enumerate(sa_solution.routes):
    if route:
        # Route from depot to first customer
        if len(route) > 0:
            first_customer = route[0]
            if first_customer == 1:
                ax4.plot([0, 10], [0, 0], color=colors[i], linewidth=2, alpha=0.7, label=f'SA Route {i+1}')
            elif first_customer == 2:
                ax4.plot([0, 12], [0, 8], color=colors[i], linewidth=2, alpha=0.7, label=f'SA Route {i+1}')
            
            # Route from customer back to depot
            if first_customer == 1:
                ax4.plot([10, 0], [0, 0], color=colors[i], linewidth=2, alpha=0.7)
            elif first_customer == 2:
                ax4.plot([12, 0], [8, 0], color=colors[i], linewidth=2, alpha=0.7)

ax4.set_xlabel('X Coordinate')
ax4.set_ylabel('Y Coordinate')
ax4.set_title('SA Solution Routes')
ax4.legend()
ax4.grid(True, alpha=0.3)

# 5. Constraint Violation Analysis
ax5 = axes[1, 1]
sa_violations = len(sa_solution.constraint_violations)
qaoa_violations = len(qaoa_solution.constraint_violations)

feasibility_data = [sa_violations, qaoa_violations]
colors5 = ['green' if v == 0 else 'red' for v in feasibility_data]

bars5 = ax5.bar(methods, feasibility_data, color=colors5, alpha=0.7)
ax5.set_ylabel('Number of Constraint Violations')
ax5.set_title('Constraint Violation Analysis')
ax5.grid(True, alpha=0.3)

# Add feasibility labels
for i, (bar, violations) in enumerate(zip(bars5, feasibility_data)):
    status = 'Feasible' if violations == 0 else 'Infeasible'
    ax5.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1, 
            status, ha='center', fontsize=10)

# 6. Performance Metrics
ax6 = axes[1, 2]
# Create performance comparison
metrics = ['Cost', 'Emissions', 'Distance', 'Feasibility']
sa_values = [sa_solution.total_cost, sa_solution.total_emissions, 
            sa_solution.total_distance, 1.0 if sa_solution.is_feasible else 0.0]
qaoa_values = [qaoa_solution.total_cost, qaoa_solution.total_emissions,
              qaoa_solution.total_distance, 1.0 if qaoa_solution.is_feasible else 0.0]

# Normalize values for comparison (except feasibility)
for i in range(3):  # Cost, Emissions, Distance
    max_val = max(sa_values[i], qaoa_values[i])
    if max_val > 0:
        sa_values[i] = (sa_values[i] / max_val) * 100
        qaoa_values[i] = (qaoa_values[i] / max_val) * 100

x = np.arange(len(metrics))
width = 0.35

ax6.bar(x - width/2, sa_values, width, label='Simulated Annealing', alpha=0.7)
ax6.bar(x + width/2, qaoa_values, width, label='QAOA', alpha=0.7)

ax6.set_xlabel('Metrics')
ax6.set_ylabel('Normalized Score')
ax6.set_title('Performance Metrics Comparison')
ax6.set_xticks(x)
ax6.set_xticklabels(metrics)
ax6.legend()
ax6.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Generate detailed analysis and insights
print("\n" + "=" * 60)
print("QUBO FORMULATION ANALYSIS AND INSIGHTS")
print("=" * 60)

print("\n1. QUBO FORMULATION RESULTS:")
print(f"  Binary variables: {instance.n_variables}")
print(f"  QUBO matrix size: {Q_matrix.shape}")
print(f"  Matrix sparsity: {(1 - np.count_nonzero(Q_matrix) / (instance.n_variables ** 2)) * 100:.1f}%")
print(f"  Symmetric matrix: {np.allclose(Q_matrix, Q_matrix.T)}")

print("\n2. QUANTUM-INSPIRED SOLVER PERFORMANCE:")
print(f"  Simulated Annealing:")
print(f"    Best energy: {sa_solution.energy:.2f}")
print(f"    Cost: ${sa_solution.total_cost:.2f}")
print(f"    Emissions: {sa_solution.total_emissions:.2f} kg CO₂")
print(f"    Feasible: {sa_solution.is_feasible}")
print(f"    Convergence: {len(sa_energy_history)} iterations")

print(f"  QAOA:")
print(f"    Best energy: {qaoa_solution.energy:.2f}")
print(f"    Cost: ${qaoa_solution.total_cost:.2f}")
print(f"    Emissions: {qaoa_solution.total_emissions:.2f} kg CO₂")
print(f"    Feasible: {qaoa_solution.is_feasible}")
print(f"    Layers: 3, Iterations: 100")

print("\n3. SOLUTION QUALITY ANALYSIS:")
sa_routes = len(sa_solution.routes)
qaoa_routes = len(qaoa_solution.routes)

print(f"  Number of routes:")
print(f"    SA: {sa_routes}, QAOA: {qaoa_routes}")
print(f"  Customers served:")
print(f"    SA: {sum(len(route) for route in sa_solution.routes)}")
print(f"    QAOA: {sum(len(route) for route in qaoa_solution.routes)}")
print(f"  Constraint violations:")
print(f"    SA: {len(sa_solution.constraint_violations)}")
print(f"    QAOA: {len(qaoa_solution.constraint_violations)}")

print("\n4. COMPUTATIONAL COMPLEXITY:")
print(f"  Problem size: {instance.n_variables} binary variables")
print(f"  Search space: 2^{instance.n_variables} = {2**instance.n_variables:,} possible solutions")
print(f"  Classical enumeration: O(2^n) complexity")
print(f"  Quantum annealing: Potential exponential speedup")
print(f"  QUBO matrix operations: O(n^2) complexity")

print("\n5. QUANTUM ADVANTAGE ANALYSIS:")
print(f"  Theoretical speedup: Exponential for large instances")
print(f"  Practical limitations:")
print(f"    - Hardware noise and decoherence")
print(f"    - Limited qubit connectivity")
print(f"    - Problem embedding challenges")
print(f"    - Hybrid classical-quantum approaches needed")

print("\n6. QUBO FORMULATION INSIGHTS:")
print(f"  Constraint handling: Penalty method with large coefficients")
print(f"  Multi-objective: Weighted sum of cost and emissions")
print(f"  Binary encoding: 3D indexing (customer, position, vehicle)")
print(f"  Matrix structure: Sparse with specific patterns")
print(f"  Optimization: Unconstrained quadratic form")

print("\n7. ALGORITHM COMPARISON:")
print(f"  Simulated Annealing:")
print(f"    ✓ Classical, well-understood")
print(f"    ✓ Easy to implement")
print(f"    ✓ Good convergence properties")
print(f"    ⚠ Can get stuck in local minima")
print(f"    ⚠ Temperature tuning required")

print(f"  QAOA:")
print(f"    ✓ Quantum-inspired algorithm")
    ✓ Theoretical quantum advantage")
    ✓ Variational approach")
    ⚠ Complex parameter optimization")
    ⚠ Limited by classical simulation")

print("\n8. PRACTICAL CONSIDERATIONS:")
print(f"  Hardware requirements:")
print(f"    - Quantum annealer (D-Wave, etc.)")
print(f"    - Gate-based quantum computer")
print(f"    - Classical simulation for testing")
print(f"  Software requirements:")
print(f"    - QUBO formulation tools")
print(f"    - Quantum programming frameworks")
print(f"    - Hybrid classical-quantum algorithms")

print("\n9. SCALABILITY ANALYSIS:")
print(f"  Current instance: {instance.n_customers} customers, {instance.n_vehicles} vehicles")
print(f"  Variables: {instance.n_variables}")
print(f"  Practical limit: ~1000 variables on current quantum hardware")
print(f"  Future prospects: Larger quantum computers")
print(f"  Hybrid approaches: Decomposition for larger problems")

print("\n10. APPLICATIONS AND USE CASES:")
print(f"  ✓ Small to medium routing problems")
print(f"  ✓ Combinatorial optimization")
print(f"  ✓ Research and prototyping")
print(f"  ✓ Hybrid classical-quantum systems")
print(f"  ✓ Quantum computing education")

print("\n11. CHALLENGES AND LIMITATIONS:")
print(f"  ⚠ Limited quantum hardware availability")
print(f"  ⚠ Noise and error correction issues")
print(f"  ⚠ Complex QUBO formulation for large problems")
print(f"  ⚠ Constraint embedding challenges")
print(f"  ⚠ Hybrid algorithm complexity")
print(f"  ⚠ Performance validation difficulties")

print("\n12. FUTURE DIRECTIONS:")
print(f"  • Improved quantum hardware with more qubits")
print(f"  • Better error correction and noise reduction")
print(f"  • Advanced QUBO formulation techniques")
print(f"  • Hybrid classical-quantum algorithms")
print(f"  • Problem-specific quantum circuits")
print(f"  • Real-world deployment and testing")

print("\n13. CONCLUSION:")
print(f"The QUBO formulation successfully demonstrates how quantum computing")
print(f"can be applied to Green Vehicle Routing Problems. While current")
print(f"quantum hardware limitations restrict practical applications, the")
print(f"theoretical foundation shows promise for exponential speedup")
print(f"in solving complex routing optimization problems.")
print(f"Hybrid classical-quantum approaches offer the most practical")
print(f"path forward for leveraging quantum advantages in real-world")
print(f"logistics optimization.")

### Key Insights from QUBO Formulation

1. **Quantum Advantage**: The QUBO formulation enables quantum computing approaches that can potentially provide exponential speedup for combinatorial optimization problems like Green VRP.

2. **Binary Encoding**: The 3D binary encoding (customer × position × vehicle) provides a systematic way to represent routing decisions as binary variables suitable for quantum optimization.

3. **Constraint Handling**: Penalty methods effectively transform constrained optimization into unconstrained QUBO form, though careful tuning of penalty weights is required.

4. **Hybrid Approaches**: Classical-quantum hybrid algorithms like QAOA and quantum-inspired simulated annealing provide practical solutions given current hardware limitations.

5. **Scalability Challenges**: Current quantum hardware limits problem size, but ongoing advances in quantum computing promise to overcome these limitations.

### When to Prefer This Tier Over Earlier Tiers

**Choose QUBO Formulation when:**
- Problem size exceeds capabilities of classical exact methods
- Quantum computing hardware is available
- Research and development of quantum algorithms is the goal
- Exponential speedup potential provides significant value
- Hybrid classical-quantum approaches can be implemented
- Future-proofing for quantum computing adoption is important

**Stick with earlier tiers when:**
- Problem instances are small enough for classical methods
- Quantum hardware is not available
- Classical methods provide acceptable performance
- Implementation simplicity is preferred
- Real-time constraints require proven classical solutions
- Quantum expertise is not available

### Summary

The QUBO formulation represents the cutting edge of optimization technology, bringing quantum computing capabilities to Green Vehicle Routing Problems. While current quantum hardware limitations restrict immediate practical applications, the theoretical foundation and quantum-inspired classical algorithms demonstrate the potential for revolutionary advances in combinatorial optimization.

The key innovation is the transformation of complex routing problems into quadratic unconstrained binary optimization form, making them suitable for quantum annealing and variational quantum algorithms. This approach opens the door to exponential computational speedup for large-scale routing problems that are intractable for classical computers.

For practical applications today, quantum-inspired classical algorithms and hybrid approaches provide the most viable path forward, offering a glimpse into the future of quantum-empowered logistics optimization while working within current technological constraints.